In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import HTML, display

In [ ]:
# ==========================================
#       CONEXIÓN A LA BASE DE DATOS
# ==========================================
USER = 'postgres.qyfemymxjmoijgrmpmwy'
PASSWORD = 'MACxnySnSKqP,76'
HOST = 'aws-1-us-east-1.pooler.supabase.com'
PORT = '6543'
DBNAME = 'postgres'
DATABASE_URI = f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}'
engine = create_engine(DATABASE_URI)

In [ ]:
# ==========================================
#      FUNCIONES DE LOS TRES GRÁFICOS
# ==========================================

# --- GRÁFICO 1: Tasa de Abandono Animada (PESTAÑA 1) ---
def generar_grafico_abandono(engine):
    # --- CONSULTA SQL: Se agrupa por año y por provincia buscando la DESERCIÓN
    query_provincias = '''
    SELECT
        anio AS Anio,
        provincia as Provincia,
        (SUM(inscritos_marzo_grado_1) + SUM(inscritos_marzo_grado_2) + SUM(inscritos_marzo_grado_3) +
         SUM(inscritos_marzo_grado_4) + SUM(inscritos_marzo_grado_5) + SUM(inscritos_marzo_grado_6) +
         SUM(inscritos_marzo_anio_1) + SUM(inscritos_marzo_anio_2) + SUM(inscritos_marzo_anio_3) +
         SUM(inscritos_marzo_anio_4) + SUM(inscritos_marzo_anio_5) + SUM(inscritos_marzo_anio_6)) AS Total_Alumnos,
        (SUM(desercion_grado_1) + SUM(desercion_grado_2) + SUM(desercion_grado_3) +
         SUM(desercion_grado_4) + SUM(desercion_grado_5) + SUM(desercion_grado_6) +
         SUM(desercion_anio_1) + SUM(desercion_anio_2) + SUM(desercion_anio_3) +
         SUM(desercion_anio_4) + SUM(desercion_anio_5) + SUM(desercion_anio_6)) AS Total_Salidas
    FROM
        trayectoria_por_sexo
    GROUP BY
        anio, provincia
    ORDER BY
        anio, provincia
    '''

    # --- EXTRACCIÓN Y CÁLCULOS ---
    df_sql = pd.read_sql_query(query_provincias, engine)
    df_sql.columns = ['Anio', 'Provincia', 'Total_Alumnos', 'Total_Salidas']

    df_animado = df_sql.copy()
    # Se calcula la tasa de abandono dividiendo salidas sobre inscritos totales
    df_animado['Tasa_Abandono'] = (df_animado['Total_Salidas'] / df_animado['Total_Alumnos']) * 100
    df_animado['Anio'] = df_animado['Anio'].astype(int)

    # Se formatean los números grandes para que el punto sea el separador de los miles
    df_animado['Alumnos_Str'] = df_animado['Total_Alumnos'].apply(lambda x: f'{x:,.0f}'.replace(',', '.'))
    df_animado['Salidas_Str'] = df_animado['Total_Salidas'].apply(lambda x: f'{x:,.0f}'.replace(',', '.'))

    # --- ORDEN Y LIMPIEZA ---
    df_animado = df_animado.sort_values(by = ['Anio', 'Provincia']).dropna(subset = ['Tasa_Abandono'])

    # --- GRÁFICO ANIMADO ---
    fig = px.bar(
        df_animado, x = 'Provincia', y = 'Tasa_Abandono', color = 'Provincia',
        animation_frame = 'Anio', animation_group = 'Provincia',
        title = '🏫 Tasa de Abandono por Provincia 🇦🇷 📉',
        range_y = [0, df_animado['Tasa_Abandono'].max() * 1.1],
        template = 'plotly_dark', hover_name = 'Provincia',
        hover_data = {
            'Provincia': False, 'Anio': False, 'Tasa_Abandono': ':.2f',
            'Alumnos_Str': True, 'Salidas_Str': True
        },
        labels = {
            'Tasa_Abandono': 'Tasa de Abandono', 'Alumnos_Str': 'Total Inscritos',
            'Salidas_Str': 'Total Deserción'
        }
    )

    # --- CONFIGURACIÓN DE DISEÑO Y CONTROLES ---
    fig.update_layout(
        title = dict(font = dict(size = 35, family = 'Amaranth'), x = 0.5, y = 0.95),
        xaxis = dict(categoryorder = 'total descending', title = '', tickangle = -45, tickmode = 'linear'),
        separators = ',.',
        paper_bgcolor = 'rgba(0, 0, 0, 0)',
        plot_bgcolor = 'rgba(0, 0, 0, 0)',
        yaxis = dict(title = 'Tasa de Abandono (%)', title_standoff = 20),
        showlegend = True,
        legend = dict(title = 'Click: Ocultar | Doble Click: Aislar', font = dict(size = 12), yanchor = 'top', y = 0.99, xanchor = 'left', x = 1.02),
        height = 800,
        margin = dict(b = 180, t = 100, l = 70, r = 50),
        transition = dict(duration = 5000),
        updatemenus = [dict(type = 'buttons', direction = 'left', showactive = False, x = 0.005, y = -0.1, xanchor = 'left', yanchor = 'top', buttons = fig.layout.updatemenus[0].buttons)]
    )

    TIEMPO_FRAME = 1200
    TIEMPO_TRANSICION = 800

    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = TIEMPO_FRAME
    fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = TIEMPO_TRANSICION

    for step in fig.layout.sliders[0].steps:
        step.args[1]['frame']['duration'] = TIEMPO_FRAME
        step.args[1]['transition']['duration'] = TIEMPO_TRANSICION

    fig.layout.sliders[0].y = -0.2
    fig.layout.sliders[0].x = 0.15
    fig.layout.sliders[0].len = 1.0
    fig.layout.sliders[0].font.size = 12
    fig.layout.sliders[0].pad = {'b': 10, 't': 15}
    fig.layout.sliders[0].currentvalue.prefix = 'Año = '

    # Se encapsula todo en un código HTML.
    return fig.to_html(full_html = False, include_plotlyjs = False, config={'displayModeBar': False})

In [ ]:
# --- GRÁFICO 2: Embudo (PESTAÑA 2A) ---
def generar_grafico_embudo(engine):
    # --- CONSULTA SQL: Se suma por un lado el total de alumnos inscritos por grado y por año y por otro las mujeres inscritas por grado y por año.
    # Se agrupan por cada año.
    query = '''
    SELECT anio,
        SUM(inscritos_marzo_grado_1) AS t_g1, SUM(inscritos_marzo_mujeres_grado_1) AS m_g1,
        SUM(inscritos_marzo_grado_2) AS t_g2, SUM(inscritos_marzo_mujeres_grado_2) AS m_g2,
        SUM(inscritos_marzo_grado_3) AS t_g3, SUM(inscritos_marzo_mujeres_grado_3) AS m_g3,
        SUM(inscritos_marzo_grado_4) AS t_g4, SUM(inscritos_marzo_mujeres_grado_4) AS m_g4,
        SUM(inscritos_marzo_grado_5) AS t_g5, SUM(inscritos_marzo_mujeres_grado_5) AS m_g5,
        SUM(inscritos_marzo_grado_6) AS t_g6, SUM(inscritos_marzo_mujeres_grado_6) AS m_g6,
        SUM(inscritos_marzo_anio_1) AS t_a1, SUM(inscritos_marzo_mujeres_anio_1) AS m_a1,
        SUM(inscritos_marzo_anio_2) AS t_a2, SUM(inscritos_marzo_mujeres_anio_2) AS m_a2,
        SUM(inscritos_marzo_anio_3) AS t_a3, SUM(inscritos_marzo_mujeres_anio_3) AS m_a3,
        SUM(inscritos_marzo_anio_4) AS t_a4, SUM(inscritos_marzo_mujeres_anio_4) AS m_a4,
        SUM(inscritos_marzo_anio_5) AS t_a5, SUM(inscritos_marzo_mujeres_anio_5) AS m_a5,
        SUM(inscritos_marzo_anio_6) AS t_a6, SUM(inscritos_marzo_mujeres_anio_6) AS m_a6,
        SUM(aprobados_grado_1) AS at_g1, SUM(aprobados_mujeres_grado_1) AS am_g1,
        SUM(aprobados_grado_2) AS at_g2, SUM(aprobados_mujeres_grado_2) AS am_g2,
        SUM(aprobados_grado_3) AS at_g3, SUM(aprobados_mujeres_grado_3) AS am_g3,
        SUM(aprobados_grado_4) AS at_g4, SUM(aprobados_mujeres_grado_4) AS am_g4,
        SUM(aprobados_grado_5) AS at_g5, SUM(aprobados_mujeres_grado_5) AS am_g5,
        SUM(aprobados_grado_6) AS at_g6, SUM(aprobados_mujeres_grado_6) AS am_g6,
        SUM(aprobados_anio_1) AS at_a1, SUM(aprobados_mujeres_anio_1) AS am_a1,
        SUM(aprobados_anio_2) AS at_a2, SUM(aprobados_mujeres_anio_2) AS am_a2,
        SUM(aprobados_anio_3) AS at_a3, SUM(aprobados_mujeres_anio_3) AS am_a3,
        SUM(aprobados_anio_4) AS at_a4, SUM(aprobados_mujeres_anio_4) AS am_a4,
        SUM(aprobados_anio_5) AS at_a5, SUM(aprobados_mujeres_anio_5) AS am_a5,
        SUM(aprobados_anio_6) AS at_a6, SUM(aprobados_mujeres_anio_6) AS am_a6
    FROM
      trayectoria_por_sexo
    GROUP BY
      anio
    ORDER BY
      anio ASC
    '''
    df_raw = pd.read_sql(query, engine) # Ejecuta la consulta y la guarda los datos crudos(raw) en un DataFrame.
    etapas_formato = [('1ro Primaria', 'g1'), ('2do Primaria', 'g2'), ('3ro Primaria', 'g3'),
                      ('4to Primaria', 'g4'), ('5to Primaria', 'g5'), ('6to Primaria', 'g6'),
                      ('1ro Secundaria', 'a1'), ('2do Secundaria', 'a2'), ('3ro Secundaria', 'a3'),
                      ('4to Secundaria', 'a4'), ('5to Secundaria', 'a5'), ('6to Secundaria', 'a6')]
    datos = []
    for _, row in df_raw.iterrows():     # iterrow devuelve el índice (que se ignora con _) y la fila.
        a = str(int(row['anio']))        # Convierte los años en texto para que no tengan decimales.
        for nom, suf in etapas_formato:
            t, m = row[f't_{suf}'], row[f'm_{suf}'] # t: total, m: mujeres. "suf" es el sufijo que se usa para nombrar a las columnas.
            at, am = row[f'at_{suf}'], row[f'am_{suf}']
            datos.append({'Anio': a, 'Etapa': nom, 'Sexo': 'Mujeres', 'Alumnos': m, 'Aprobados': am})
            datos.append({'Anio': a, 'Etapa': nom, 'Sexo': 'Varones', 'Alumnos': t - m, 'Aprobados': at - am}) # Se resta el total con las mujeres para obtener a los varones.

    df = pd.DataFrame(datos)
    # Se extraen sólo los datos para el slider y se ordenan.
    anios = sorted(df['Anio'].unique().tolist()) #"unique()" devuelve los valores únicos de una columna, "tolist()" convierte la lista en texto, "sorted" ordena la lista.
    color_map = {'Varones': '#B9B9B9', 'Mujeres': '#C9AAFC'}

    fig = go.Figure()
    for a in anios:
        df_a = df[df['Anio'] == a]
        for s, icono in [('Varones', '👨'), ('Mujeres', '👩')]:
            df_s = df_a[df_a['Sexo'] == s]

            fig.add_trace(go.Funnel(
                name = f'{icono} {s}',
                y = df_s['Etapa'],
                x = df_s['Alumnos'],
                customdata = df_s['Aprobados'],
                marker = dict(color = color_map[s]),
                visible = (a == anios[0]), # Empieza desde el primer año (2011).
                texttemplate = '%{value:,.0f}', textfont = dict(family = 'Amaranth', size = 18), #"texttemplate" permite mostrar el valor de cada barra. Agarra el valor de cada barra y le saca los decimales.
                hovertemplate = (f'<b>{icono} {s}</b><br>' +
                                 f'<b>Etapa:</b> %{{y}}<br>' +
                                 f'<b>Inscriptos:</b> %{{x:,.0f}}<br>' +
                                 f'<b>Aprobados:</b> %{{customdata:,.0f}}<extra></extra>')
            ))

    # Se crea una lista que guarda los años que se van a ir mostrando en el slider a medida que se seleccionen.
    pasos = []
    for i, a in enumerate(anios):
        vis = [False] * (len(anios) * 2)
        vis[i * 2], vis[i * 2 + 1] = True, True
        paso = dict(method = 'update', label = str(a), args = [{'visible': vis}, {'title': f'Flujo de Retención Escolar: Varones vs. Mujeres ({a})'}])
        pasos.append(paso)

    fig.update_layout(
        template = 'plotly_dark', funnelmode = 'stack', separators = ',.',
        paper_bgcolor = 'rgba(0, 0, 0, 0)', # Se transparenta el lienzo, en donde va el gráfico.
        plot_bgcolor = 'rgba(0, 0, 0, 0)',  # Se transparenta el fondo del gráfico.
        font = dict(family = 'Amaranth', color = 'white', size = 16),
        title = dict(font = dict(family = 'Amaranth', size = 26), x = 0.5, y = 0.95),
        margin = dict(t = 80, b = 120, l = 10, r = 10), height = 750, # Se achican los márgenes para que el embudo se vea más grande.
        sliders = [dict(
            active = 0, currentvalue = {'prefix': 'Año visualizado: ', 'font': {'size': 20, 'family': 'Amaranth'}},
            pad = {'t': 30}, steps = pasos, bgcolor = 'rgba(20, 31, 45, 0.9)', bordercolor = '#8fd8fb', # Pad es el espacio entre los títulos de los ejes y el slider.
            font = dict(family = 'Amaranth', color = 'white', size = 13),
            len = 0.85, x = 0.5, y = 0.05, xanchor = 'center' # Se achica y centra el slider.
        )]
    )
    # Se encapsula todo en un código HTML.
    return fig.to_html(full_html = False, include_plotlyjs = False, config={'displayModeBar': False})

In [ ]:
# --- GRÁFICO 3: Mancuernas (PESTAÑA 2B) ---
def generar_grafico_mancuernas(engine):
    # --- CONSULTA SQL: Se suma los inscritos en marzo por grado y año, las mujeres inscritas en marzo por grado y año,
    # el total de los aprobados por grado y año, y el total de las mujeres aprobadas por grado y año.
    query = '''
    SELECT anio,
        SUM(inscritos_marzo_grado_1) AS ins_t_g1, SUM(inscritos_marzo_mujeres_grado_1) AS ins_m_g1,
        SUM(aprobados_grado_1) AS apr_t_g1, SUM(aprobados_mujeres_grado_1) AS apr_m_g1,
        SUM(inscritos_marzo_grado_6) AS ins_t_g6, SUM(inscritos_marzo_mujeres_grado_6) AS ins_m_g6,
        SUM(aprobados_grado_6) AS apr_t_g6, SUM(aprobados_mujeres_grado_6) AS apr_m_g6,
        SUM(inscritos_marzo_anio_1) AS ins_t_a1, SUM(inscritos_marzo_mujeres_anio_1) AS ins_m_a1,
        SUM(aprobados_anio_1) AS apr_t_a1, SUM(aprobados_mujeres_anio_1) AS apr_m_a1,
        SUM(inscritos_marzo_anio_6) AS ins_t_a6, SUM(inscritos_marzo_mujeres_anio_6) AS ins_m_a6,
        SUM(aprobados_anio_6) AS apr_t_a6, SUM(aprobados_mujeres_anio_6) AS apr_m_a6
    FROM trayectoria_por_sexo GROUP BY anio ORDER BY anio ASC
    '''
    df_sql = pd.read_sql(query, engine)
    etapas = [('Inicio Primaria', 'g1'), ('Fin Primaria', 'g6'), ('Inicio Secundaria', 'a1'), ('Fin Secundaria', 'a6')]
    # Se crea una lista con las tasas de aprobación (porcentajes) y las cantidades absolutas de inscritos y aprobados por sexo, año y etapa.
    datos = []
    for _, row in df_sql.iterrows():
        anio_limpio = str(int(row['anio'])) # Transforma los años en texto.
        for nom, suf in etapas:
            im, am = row[f'ins_m_{suf}'], row[f'apr_m_{suf}']
            iv, av = row[f'ins_t_{suf}'] - im, row[f'apr_t_{suf}'] - am
            tm = (am/im) * 100 if im > 0 else 0
            tv = (av/iv) * 100 if iv > 0 else 0
            datos.append({
                'Anio': anio_limpio, 'Etapa': nom,
                'Tasa_Varones': tv, 'Tasa_Mujeres': tm,
                'Apr_V': av, 'Ins_V': iv,
                'Apr_M': am, 'Ins_M': im
            })

    df = pd.DataFrame(datos)
    anios = sorted(df['Anio'].unique().tolist())
    color_v = '#B9B9B9'
    color_m = '#C9AAFC'

    fig = go.Figure()
    for a in anios:
        df_f = df[df['Anio'] == a].iloc[::-1]
        vis = (a == anios[0])
        for _, r in df_f.iterrows():
            fig.add_trace(go.Scatter(
                x = [r['Tasa_Varones'], r['Tasa_Mujeres']], y = [r['Etapa'], r['Etapa']],
                mode = 'lines', line = dict(color = 'rgba(255, 255, 255, 0.2)', width = 4),
                showlegend = False, visible = vis, hoverinfo = 'skip'
            ))
            fig.add_trace(go.Scatter(
                x = [r['Tasa_Varones']], y = [r['Etapa']], mode = 'markers',
                marker = dict(color = color_v, size = 16), name = '👨 Varones',
                legendgroup = 'v', showlegend = False, visible = vis,
                customdata = [[r['Apr_V'], r['Ins_V']]], # Pasamos los datos crudos
                hovertemplate = f'<b>👨 Varones</b><br>Etapa: %{{y}}<br>Tasa: %{{x:.1f}}%<br>Aprobados: %{{customdata[0]:,.0f}} de %{{customdata[1]:,.0f}}<extra></extra>'
            ))
            fig.add_trace(go.Scatter(
                x = [r['Tasa_Mujeres']], y = [r['Etapa']], mode = 'markers',
                marker = dict(color = color_m, size = 16), name = '👩 Mujeres',
                legendgroup = 'm', showlegend = False, visible = vis,
                customdata = [[r['Apr_M'], r['Ins_M']]], # Pasamos los datos crudos
                hovertemplate = f'<b>👩 Mujeres</b><br>Etapa: %{{y}}<br>Tasa: %{{x:.1f}}%<br>Aprobados: %{{customdata[0]:,.0f}} de %{{customdata[1]:,.0f}}<extra></extra>'
            ))

    # Se configuran los pasos del slider. Como cada año requiere 12 capas de dibujo (trazas), el bucle calcula matemáticamente cuáles 12 hacer visibles y cuáles ocultar para el año seleccionado.
    pasos = []
    trazas_por_anio = 12
    for i, a in enumerate(anios):
        v = [False] * len(fig.data)
        for j in range(i * trazas_por_anio, (i + 1) * trazas_por_anio): v[j] = True
        pasos.append(dict(method = 'update', label = str(a), args = [{'visible': v}, {'title': f'Brecha de Promoción por Sexo - Año {a}'}]))

    fig.update_layout(
        template = 'plotly_dark', height = 750, separators = ',.', font = dict(family = 'Amaranth', size = 16),
        paper_bgcolor = 'rgba(0, 0, 0, 0)', # Se transparenta el lienzo, en donde va el gráfico.
        plot_bgcolor = 'rgba(0, 0, 0, 0)',  # Se transparenta el fondo del gráfico.
        title = dict(text = f'Brecha de Promoción por Sexo - Año {anios[0]}', font = dict(family = 'Amaranth', size = 26), x = 0.5, y = 0.95),
        xaxis = dict(title = 'Tasa de Aprobación (%)', range = [0, 105], ticksuffix = '%', gridcolor = 'rgba(255, 255, 255, 0.05)'),
        yaxis = dict(gridcolor = 'rgba(255, 255, 255, 0.05)', ticklabelstandoff = 20),
        margin = dict(t = 80, b = 120, l = 10, r = 10),
        sliders = [dict(
            active = 0, steps = pasos, y = -0.1,
            len = 1.0, x = 0.45, xanchor = 'center',
            currentvalue = {'prefix': 'Año visualizado: ', 'font': {'size': 20, 'family': 'Amaranth'}},
            bgcolor = 'rgba(20, 31, 45, 0.9)', bordercolor = '#8fd8fb',
            font = dict(family = 'Amaranth', color = 'white', size = 15)
        )]
    )
    # Se encapsula todo en un código HTML.
    return fig.to_html(full_html = False, include_plotlyjs = False, config={'displayModeBar': False})

In [ ]:
# ==========================================
#    ARMADO DE LAS VARIABLES PARA EL HTML
# ==========================================

# Pestaña 1: Gráfico de abandono escolar.
grafico_1_html = generar_grafico_abandono(engine)

In [ ]:
# Pestaña 2: Menú integrado con embudo y mancuernas.
html_embudo = generar_grafico_embudo(engine)
html_mancuernas = generar_grafico_mancuernas(engine)

grafico_2_html = f"""
<div style="display: flex; gap: 10px; margin-bottom: 20px; justify-content: center;">
    <button id="btn-embudo" onclick="mostrarSubGrafico('embudo')" style="background-color: #eab308; color: #060e24; padding: 8px 16px; border-radius: 6px; font-weight: bold; cursor: pointer; border: none; transition: 0.3s;">📉 Flujo de Retención</button>
    <button id="btn-mancuernas" onclick="mostrarSubGrafico('mancuernas')" style="background-color: #1e293b; color: #94a3b8; padding: 8px 16px; border-radius: 6px; font-weight: bold; cursor: pointer; border: 1px solid #334155; transition: 0.3s;">⚖️ Brecha de Promoción</button>
</div>

<div id="cont-embudo" style="display: block; width: 100%;">
    {html_embudo}
</div>
<div id="cont-mancuernas" style="display: none; width: 100%;">
    {html_mancuernas}
</div>

<script>
    function mostrarSubGrafico(grafico) {{
        if (grafico === 'embudo') {{
            document.getElementById('cont-embudo').style.display = 'block';
            document.getElementById('cont-mancuernas').style.display = 'none';
            document.getElementById('btn-embudo').style.backgroundColor = '#eab308';
            document.getElementById('btn-embudo').style.color = '#060e24';
            document.getElementById('btn-mancuernas').style.backgroundColor = '#1e293b';
            document.getElementById('btn-mancuernas').style.color = '#94a3b8';
        }} else {{
            document.getElementById('cont-embudo').style.display = 'none';
            document.getElementById('cont-mancuernas').style.display = 'block';
            document.getElementById('btn-mancuernas').style.backgroundColor = '#eab308';
            document.getElementById('btn-mancuernas').style.color = '#060e24';
            document.getElementById('btn-embudo').style.backgroundColor = '#1e293b';
            document.getElementById('btn-embudo').style.color = '#94a3b8';
        }}
        window.dispatchEvent(new Event('resize'));
    }}
</script>
"""